# 31. Season-end OVR progression

After a simulated season (notebook 30's story), every player gets a **new OVR for next season**: up, down or the
same. This is also the year-to-year engine a multi-season franchise mode will use later.

**The idea in one line:** the new rating is **the same projection that made today's ratings, run one season later**,
with the simulated 2026-27 season added to each player's history. Nothing new is fitted. `src/progression.py` calls
the same steps:

- **NHL skaters:** notebook 18 (his last three seasons, weighted, pulled toward his own prior, aged on the curve);
- **NHL goalies:** notebook 21 (the same, with the goalie prior);
- **prospects and new players:** notebooks 19, 21 and 22 (pre-NHL scoring or save%, draft position, age);
- **OVR:** notebook 20's locked scale (goalies on their own anchors).

The fitted pieces (the priors, their centering shifts, the aging curve) are built once from real history and then
**stay frozen**, like the OVR scale. Refitting them on simulated seasons would let the whole league drift.

This notebook writes nothing.

## How a rating moves, in plain English

Going into 2027-28, a player's projection is a blend of his last three seasons (2024-25, 2025-26 and the simulated
2026-27), weighted by minutes, plus a pull toward his prior. Three things move it:

1. **His simulated season.** A season above his projection pulls the blend up, below pulls it down. The more minutes
   he played, the more it counts. This is the **season part**.
2. **Aging.** Every season in the blend is moved to his new age along the aging curve, and his prior is one year
   older with one more season of experience.
3. **The window rolls.** His 2023-24 season drops out of the three. If it was better than his projection, losing it
   costs him; if it was worse, he gains.

Two and three together are the **aging part**: what would happen if his season landed exactly on his projection.

**Players with no simulated NHL minutes** (scratched all season, in the AHL, sitting out) get no 2026-27 season. NHL
players still roll their window, regress more toward their prior and age. **The simulator plays NHL games only: no
AHL or junior season is simulated**, so a prospect who doesn't dress has nothing new. **Ryan's rule (adopted after the
first version of this notebook):** such a prospect gets his OVR before + **the real average change of a prospect year
at his age and position** (section 7), flagged "expected development, not simulated". The engine's own number (same
pre-NHL stats one year older) stays in its own column, `engine_ovr_after`.

In [1]:
import sys
sys.path.append("../src")

import subprocess
import time
from pathlib import Path
import numpy as np
import pandas as pd

import franchise as FR
import progression as PG

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
BUNDLE = DATA / "simulator_2026-27"
TRADES = DATA / "trades_2026-27.csv"
CACHE = BUNDLE / "updated"
OPENING = "2026-09-29"
TEAM = "DET"
SIT_OUT = list(pd.read_csv(DATA / "sit_out_2026-27.csv")["playerId"])
MAIN_SEASONS = [2018, 2021, 2022, 2023, 2024, 2025]     # full 82-game seasons t (rating going into t vs. into t+1)
SHORT = [2019, 2020]                                     # 2019-20 stopped in March, 2020-21 had 56 games
SPEED = {}

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

clock = time.time()
E = PG.load_engine(DATA)
SPEED["build the engine (once)"] = time.time() - clock
print(f"Engine built in {SPEED['build the engine (once)']:.0f} s: {len(E['before']):,} players rated today, "
      f"D centering shift {E['sk']['shift']}, goalie shift {E['g_model']['shift']:+.4f} GAR/60 (both frozen)")

3 pick(s) without a playerId dropped
Engine built in 19 s: 2,503 players rated today, D centering shift {'D': np.float64(0.05125824632175369)}, goalie shift -0.0239 GAR/60 (both frozen)


## 1. Check first: the engine reproduces today's ratings exactly

Run on the real history that made today's ratings (seasons through 2025-26), `project_season` must give today's
2026-27 projection (`forecast.current_projections()`) and today's OVR (`ratings_2026-27.csv`) for every player:
NHL skaters, NHL goalies, skater and goalie prospects, and the new players from notebook 22.

In [2]:
rep = PG.reproduce_today(E)
print(f"Players: {rep['players']:,}; missing {len(rep['missing'])}, extra {len(rep['extra'])}")
print(f"Largest projection gap: {rep['max_proj_gap']:.1e} WAR/60; largest rating gap {rep['max_rating_gap']:.1e}; "
      f"OVRs that differ: {rep['ovr_differ']}")
print("Largest gap by source (WAR/60):")
print(rep["by_source"].map(lambda x: f"{x:.1e}").to_string())
assert not rep["missing"] and not rep["extra"] and rep["max_proj_gap"] < 1e-12 and rep["max_rating_gap"] < 1e-12
assert rep["ovr_differ"] == 0
print("Reproduced: every player, every OVR (gaps of 1e-16 are the computer's rounding, not a difference)")

Players: 2,503; missing 0, extra 0
Largest projection gap: 9.9e-17 WAR/60; largest rating gap 9.9e-17; OVRs that differ: 0
Largest gap by source (WAR/60):
source
NHL goalie (nb21)         9.7e-17
NHL skater (nb18)         9.9e-17
goalie prospect (nb21)    9.7e-17
new to data (nb22)        7.6e-17
skater prospect (nb19)    9.9e-17
Reproduced: every player, every OVR (gaps of 1e-16 are the computer's rounding, not a difference)


## 2. One story: a full season from opening night

Notebook 30's story, played from opening night with preseason ratings (seed 0), then every player's 2027-28 OVR.

**Columns:** `ovr_before` is today's 2026-27 rating, `ovr_after` the new one, `change` the whole-number change.
`aging_part` and `season_part` are in exact (unrounded) OVR, so they add up to the exact change, which can differ from
the whole-number change by rounding. `age` is his age in 2026-27 (February 1, 2027). `minutes` is his 2026-27 NHL
minutes; `season_war_60` his simulated season (projection + his random draw); `season_line` is from the story.

In [3]:
inp_open = FR.load_inputs(BUNDLE, TRADES, OPENING)
clock = time.time()
story = FR.story(inp_open, TEAM, seed=0)
SPEED["one story (notebook 30)"] = time.time() - clock
clock = time.time()
prog = PG.story_progression(E, story)
SPEED["one story's progression"] = t_prog = time.time() - clock
print(f"Story: {story['range']['games']} games, {TEAM} {story['record']['wins']}-{story['record']['losses']}-"
      f"{story['record']['otl']}, {story['record']['points']} points; progression in {t_prog:.1f} s")
print("\nWho gets what kind of change:")
print(prog["flag"].replace("", "played NHL minutes in 2026-27").value_counts().to_string())
played = prog[prog["minutes"].notna()]
print("\nPlayers with NHL minutes, whole-number change:")
print(played.groupby("group")["change"].agg(["size", "mean", "std", "min", "max"]).round(2).to_string())

Story: 1344 games, DET 36-31-17, 89 points; progression in 1.9 s

Who gets what kind of change:
flag
prospect, no NHL minutes: expected development (the real average change for his age and position), not simulated    1153
played NHL minutes in 2026-27                                                                                        640
no NHL minutes: aging and regression only (no AHL or junior stats simulated)                                         570
left the projection: no NHL season in the last three                                                                 140

Players with NHL minutes, whole-number change:
       size  mean   std  min  max
group                            
D       192   0.4  3.04   -7   11
F       384 -0.05   3.1  -11   12
G        64 -0.38  2.39   -6    6


In [4]:
COLS = ["name", "team", "position", "age", "ovr_before", "ovr_after", "change", "aging_part", "season_part",
        "minutes", "proj_war_60", "season_war_60", "season_line"]
up, down = PG.risers_fallers(prog, 10)
print("Top 10 risers:")
print(up[COLS].round(3).to_string(index=False))
print("\nTop 10 fallers:")
print(down[COLS].round(3).to_string(index=False))

Top 10 risers:
               name team position  age  ovr_before  ovr_after  change  aging_part  season_part  minutes  proj_war_60  season_war_60     season_line
       Nick Cousins  OTT        C 33.0          66         78      12       4.559        7.165  899.805       -0.013          0.096 84 GP, 14-15-29
        Noah Warren  ANA        D 22.0          69         80      11       6.743        4.092 1629.081        0.023          0.047  84 GP, 4-10-14
  William Wallinder  DET        D 24.0          72         82      10       4.286        5.828 1271.489        0.023          0.073  84 GP, 6-19-25
Guillaume Brisebois  VAN        D 29.0          68         78      10      -0.119        9.537 1350.246       -0.005          0.055  84 GP, 3-28-31
    Anton Johansson  DET        D 22.0          74         83       9       3.994        4.846 1376.075        0.023          0.076  84 GP, 5-24-29
         Kirby Dach  MTL        C 26.0          79         87       8       0.355        7.914 10

**Hockey sense: why each one moved** (one sentence each, from the table above):

Risers:
1. **Nick Cousins (33, +12):** his awful 2023-24 (-0.098 WAR/60 over 841 minutes) drops out of his window (aging part +4.6), and a 14-15-29 season at 0.096 against a -0.013 projection is a career year for a depth forward (+7.2).
2. **Noah Warren (22, +11):** a defenseman with no NHL history, rated 69 on the prospect ranking prior while the simulator played him at the position average; once he plays 1,629 minutes he's rated on his NHL projection (+6.7, the path switch below), and a solid season adds +4.1.
3. **William Wallinder (24, +10):** the same path switch for a Detroit defenseman (+4.3), plus a 6-19-25 season at 0.073, three times his projection (+5.8).
4. **Guillaume Brisebois (29, +10):** only 47 NHL minutes in the last three seasons, so his rating rested on his prior; 1,350 simulated minutes at 0.055 rewrite it (+9.5).
5. **Anton Johansson (22, +9):** another Detroit prospect defenseman: path switch +4.0 and a 5-24-29 season at 0.076 (+4.8).
6. **Kirby Dach (26, +8):** a 35-42-77 breakout, 0.247 WAR/60 against a 0.031 projection, the biggest gap between season and projection among the risers (+7.9).
7. **Quinton Burns (21, +8):** prospect defenseman: path switch +5.1, and a season a little above the position average (+3.0).
8. **Jonathan Castagna (21, +8):** a newcomer with no NHL history whose 13-18-31 season at 0.149 is five times his 0.030 projection (+6.4).
9. **Alexander Wennberg (32, +8):** his weak 2023-24 (-0.029) drops out (+1.8), and 0.081 over 1,589 minutes is far above his 0.008 projection (+5.4).
10. **Artyom Levshunov (21, +7):** after two rough real seasons (-0.093 and -0.056), a 4-36-40 season at 0.046 over 1,569 minutes pulls his blend up (+6.9).

Fallers:
1. **Kenny Connors (23, -11):** 17 real NHL minutes, so his 75 came almost entirely from his prior; a -0.094 season over 864 minutes (12-14-26) is all the NHL evidence he has now (-11.6).
2. **Jonathan Marchessault (36, -10):** age 36 plus his strong 2023-24 (0.101) dropping out cost -5.9, and a below-replacement season (-0.018) adds -3.6.
3. **William Stromgren (23, -9):** only 22 real NHL minutes, then a -0.060 season over 1,011 minutes (7-12-19): little history means a big move (-10.0).
4. **Vinni Lettieri (31, -8):** his first heavy workload (1,030 minutes, vs. 441 and 351 before) comes in at -0.073 with 6-5-11 (-8.9).
5. **Jordan Martinook (34, -8):** aging at 34 and his good 2023-24 (0.061) leaving the window cost -3.2, and a -0.047 season adds -4.8.
6. **Dominic James (24, -8):** 526 real minutes at 0.098 made him an 81; a -0.063 season over 970 minutes (4-8-12) undoes most of it (-8.1).
7. **Ryan O'Reilly (35, -7):** three steady real seasons (0.06 to 0.10), then a -0.058 draw over 1,705 minutes; his 18-33-51 is fine because points follow his minutes and role more than his WAR draw (-6.8).
8. **Tyson Hinds (23, -7):** 94 real NHL minutes, then -0.036 over 1,377 minutes as a regular (-7.8).
9. **Mario Ferraro (28, -7):** a full-time defenseman (1,585 minutes) at -0.068 against a 0.017 projection (-7.1).
10. **Brayden Schenn (35, -7):** a 4-9-13 season at -0.086 against a 0.011 projection, at an age where the curve is already pulling down (-6.6).

**The pattern:** the biggest moves go to players with **little NHL history** (their rating rested on their prior, so a full season of minutes moves it a lot), to **veterans** (aging plus a good old season leaving the window), and to **prospect defensemen** who switch paths. Stars barely move: Raymond +2 and DeBrincat -1 in this story, because three full seasons of minutes keep their blend steady.

In [5]:
print(f"{TEAM} after this season (today's organization, highest new OVR first):")
print(PG.team_roster(prog, TEAM)[COLS + ["flag"]].head(30).round(3).to_string(index=False))

DET after this season (today's organization, highest new OVR first):
                   name team position  age  ovr_before  ovr_after  change  aging_part  season_part  minutes  proj_war_60  season_war_60               season_line                                                                                                             flag
          Lucas Raymond  DET        L 24.0          90         92       2      -0.224        1.529 1549.723        0.114          0.161           84 GP, 33-57-90                                                                                                                 
         Alex DeBrincat  DET        R 29.0          91         90      -1       0.276       -0.712 1513.264        0.119          0.097           84 GP, 33-28-61                                                                                                                 
           Dylan Larkin  DET        C 30.0          88         85      -3      -1.562       -1.455 1654.72

## 3. Checks: the parts, the shortcut, and zero change

- **The parts add up:** aging part + season part = the exact change, for every player.
- **The straight-line shortcut:** lineups are fixed for the season, so each player's minutes, experience and prior are
  fixed too, and his new rating is a straight line in his season WAR/60. Two re-projections give every player's line,
  so 10,000 seasons don't need 10,000 re-projections. Checked against the direct re-projection for this story.
- **Zero change:** with aging switched off and the prior held at its 2026-27 value, a season exactly at his projection
  must change nothing, as long as no season drops out of his window (he had no 2023-24 NHL season). Players who did
  play in 2023-24 do change: that's the window piece, shown separately so it isn't hidden in "aging".

In [6]:
gap = (played["aging_part"] + played["season_part"] - played["exact_change"]).abs().max()
print(f"Parts add up: largest gap {gap:.1e} OVR over {len(played):,} players")
assert gap < 1e-9

dr = story["draws"].set_index("playerId")
war60 = dr.loc[dr["dressed"], "season_war60"]
clock = time.time()
maps = PG.linear_maps(E, story["run"])
lin = PG.apply_maps(maps, war60)
direct = PG.project_season(E, PG.NEXT, PG.season_rows(story["run"], war60, PG.season_minutes(story["run"]), E))
direct = direct.set_index("playerId")["rating_war_60"]
d = (lin.reindex(direct.index) - direct).abs().max()
print(f"Shortcut vs. direct re-projection: largest gap {d:.1e} WAR/60 over {direct.notna().sum():,} players")
assert d < 1e-12

z = PG.zero_change_check(E, story["run"])
keep = z[~z["lost_2023"]]
print(f"\nZero change: {len(keep)} players lose no season from their window "
      f"({keep['dressed'].sum()} dressed, {(~keep['dressed']).sum()} not); largest change {keep['change_war_60'].abs().max():.1e} WAR/60")
assert keep["change_war_60"].abs().max() < 1e-12
w = z[z["lost_2023"]]
print(f"The window piece ({len(w)} players whose 2023-24 season drops out; no aging, season at projection), exact OVR:")
print(w.groupby("group")["change_ovr"].describe()[["count", "mean", "std", "min", "max"]].round(2).to_string())

Parts add up: largest gap 0.0e+00 OVR over 640 players
Shortcut vs. direct re-projection: largest gap 2.3e-16 WAR/60 over 2,363 players

Zero change: 306 players lose no season from their window (89 dressed, 217 not); largest change 2.8e-17 WAR/60
The window piece (882 players whose 2023-24 season drops out; no aging, season at projection), exact OVR:
       count  mean   std   min   max
group                               
D      279.0  0.48  1.93 -5.05  9.63
F      519.0  0.39  2.04 -7.04  8.30
G       84.0 -0.01  1.37 -3.30  2.69


**The window piece:** 882 players lose a 2023-24 season. With no aging and a season exactly at projection, their
rating still moves: on average +0.48 OVR for defensemen and +0.39 for forwards (SD about 2), about 0 for goalies
(SD 1.4), from -7 to +10 for single players. It's part of the aging part in the tables above, because it's not about
how the 2026-27 season went. The 306 players with no 2023-24 season change by exactly 0 (largest 2.8e-17 WAR/60), so
the season is added with the right weight and nothing else moves.

**A path switch, not aging.** Some of the biggest aging parts belong to players who were on the **prospect path**
going into 2026-27 and dressed in the simulation. Defensemen there were rated on notebook 19's ranking prior
(`d_rank_prior`), while the simulator played them at the position average projection. After a season of NHL minutes
they're on the NHL path, rated on their projection like everyone else, and the gap shows up in the aging part.

In [7]:
sw = prog[(prog["path_before"] == "prospect") & prog["minutes"].notna()]
print(f"Dressed players who were on the prospect path: {len(sw)}")
print(sw.groupby("group")[["aging_part", "season_part", "change"]].agg(["size", "mean"]).round(2).to_string())
print(sw.sort_values("aging_part", ascending=False)[["name", "team", "position", "age", "ovr_before", "ovr_after", "change",
                                                    "aging_part", "season_part", "proj_war_60", "season_war_60"]].head(8).round(3).to_string(index=False))

Dressed players who were on the prospect path: 22
      aging_part       season_part       change      
            size  mean        size  mean   size  mean
group                                                
D              7  4.08           7  3.29      7  7.57
F             15  0.60          15  0.53     15  0.93
             name team position  age  ovr_before  ovr_after  change  aging_part  season_part  proj_war_60  season_war_60
      Noah Warren  ANA        D 22.0          69         80      11       6.743        4.092        0.023          0.047
    Quinton Burns  STL        D 21.0          73         81       8       5.117        3.017        0.023          0.050
  William Trudeau  NSH        D 24.0          71         76       5       4.672       -0.152        0.023          0.022
William Wallinder  DET        D 24.0          72         82      10       4.286        5.828        0.023          0.073
  Anton Johansson  DET        D 22.0          74         83       9       3

## 4. One story from today, with the real games so far

The same, started today (2026-11-16) with the in-season update (notebook 29's cache, through 2026-11-15) and the
sit-outs (Larkin, Edvinsson).

**How the real games so far count.** A dressed player's 2026-27 season is his **drawn season WAR/60 applied to the
whole season**, real games included: minutes = his real minutes so far + his simulated minutes, all at the drawn rate.
His real WAR so far isn't used, for two reasons:

- **the update already used the real games:** with the switch on, his draw comes from notebook 28's update, which has
  already seen his real goals and assists so far (and the "whole" version carries that draw through the season);
- **there's no real WAR so far:** the WAR model runs on finished MoneyPuck seasons, not on a partial 2026-27.

So a hot start shows up through the updated draw, not as a separate block of real WAR. A player with real games who
isn't in a lineup now (sitting out, sent down) gets no 2026-27 season at all; he's flagged.

In [8]:
inp_now = FR.load_inputs(BUNDLE, TRADES, "2026-11-16", use_update=True, cache_root=CACHE)
print("Ratings:", inp_now["ratings"])
story_now = FR.story(inp_now, TEAM, sit_out=SIT_OUT, seed=0)
prog_now = PG.story_progression(E, story_now)
cols = COLS[:9] + ["real_minutes", "minutes", "season_war_60", "season_line", "flag"]
print(f"{TEAM}, from today:")
print(PG.team_roster(prog_now, TEAM)[cols].head(26).round(3).to_string(index=False))
print("\nFlags, league:")
print(prog_now["flag"].replace("", "played NHL minutes in 2026-27").value_counts().to_string())

Ratings: updated through 2026-11-15 (notebook 29)
DET, from today:
                 name team position  age  ovr_before  ovr_after  change  aging_part  season_part  real_minutes  minutes  season_war_60              season_line                                                                                                                                                           flag
       Alex DeBrincat  DET        R 29.0          91         91       0       0.267        0.166       412.567 1604.641          0.124          84 GP, 31-37-68                                                                                                                                                               
        Lucas Raymond  DET        L 24.0          90         89      -1      -0.215       -0.807       416.083 1637.169          0.091          84 GP, 28-43-71                                                                                                                                        

## 5. 10,000 seasons: expected change and the chance of a big move

The same full season played 10,000 times (opening night, preseason ratings, seed 0), with the straight-line maps:
each player's **expected whole-number change**, the **chance of +3 or more** and of **-3 or worse**, and the 10th to
90th percentile of his new OVR. Players who don't dress have one fixed answer (aging and regression only).

In [9]:
clock = time.time()
many = PG.progression_many(E, inp_open, TEAM, n_sims=10000, seed=0)
SPEED["10,000 seasons' progression (incl. the 10,000-season run)"] = t_many = time.time() - clock
mt = many["table"]
print(f"10,000 seasons in {t_many:.1f} s")
MC = ["name", "team", "position", "age", "ovr_before", "ovr_at_projection", "expected_change", "pct_up3", "pct_down3",
      "ovr_after_p10", "ovr_after_p90", "minutes"]
dressed = mt[mt["dressed"]]
print("\nDressed players, average over players:")
print(dressed.groupby("group")[["expected_change", "pct_up3", "pct_down3"]].mean().round(2).to_string())
print("\nBiggest expected risers (dressed):")
print(dressed.nlargest(10, "expected_exact_change")[MC].round(2).to_string(index=False))
print("\nBiggest expected fallers (dressed):")
print(dressed.nsmallest(10, "expected_exact_change")[MC].round(2).to_string(index=False))

10,000 seasons in 12.3 s

Dressed players, average over players:
       expected_change  pct_up3  pct_down3
group                                     
D                 0.06    16.73      16.98
F                -0.20    15.90      19.26
G                 0.01    17.56      16.45

Biggest expected risers (dressed):
             name team position  age  ovr_before  ovr_at_projection  expected_change  pct_up3  pct_down3  ovr_after_p10  ovr_after_p90  minutes
        Bo Groulx  TOR        C 26.0          63                 69             6.19    73.83       6.35           62.0           76.0  1182.71
      Noah Warren  ANA        D 22.0          69                 76             6.71    81.10       2.12           70.0           81.0  1629.08
 Alexandre Texier  MTL        L 27.0          73                 80             5.91    84.60       0.89           74.0           83.0  1051.28
   Scott Laughton  LAK        C 32.0          73                 79             5.60    86.59       0.19    

In [10]:
print(f"{TEAM}, 10,000 seasons from opening night (dressed players):")
print(mt[(mt["team"] == TEAM) & mt["dressed"]].sort_values("ovr_before", ascending=False)[MC].round(2).to_string(index=False))
many_now = PG.progression_many(E, inp_now, TEAM, n_sims=10000, seed=0, sit_out=SIT_OUT)
mn = many_now["table"]
print(f"\n{TEAM}, 10,000 seasons from today with the update (dressed players):")
print(mn[(mn["team"] == TEAM) & mn["dressed"]].sort_values("ovr_before", ascending=False)[MC].round(2).to_string(index=False))

DET, 10,000 seasons from opening night (dressed players):
                   name team position  age  ovr_before  ovr_at_projection  expected_change  pct_up3  pct_down3  ovr_after_p10  ovr_after_p90  minutes
         Alex DeBrincat  DET        R 29.0          91                 91             0.16     7.19       2.70           89.0           93.0  1513.26
          Lucas Raymond  DET        L 24.0          90                 90             0.04     5.40       3.42           88.0           92.0  1549.72
           Dylan Larkin  DET        C 30.0          88                 86            -1.57     1.45      27.22           85.0           88.0  1654.72
       Viktor Arvidsson  DET        L 33.0          83                 83            -0.19     4.12       5.01           81.0           85.0  1226.83
          Moritz Seider  DET        D 25.0          83                 84             1.22    10.64       0.00           83.0           86.0  2015.15
            John Gibson  DET        G 33.0

## 6. Calibration against real history (the main test)

**Real changes:** the same engine on real history. For each season t, the rating going into t vs. going into t+1,
both on the locked scale with the same frozen models. The last pair (into 2025-26 vs. into 2026-27) ends at today's
ratings exactly. The frozen models were fit on all seasons, so this measures **how big the engine's changes are when
fed real seasons**, not forecast skill.

**Simulated changes:** the 10,000 seasons above (opening night, preseason ratings).

**Same population both ways:** NHL-path players who played in season t: 200+ minutes for skaters, 500+ for goalies
(simulated: everyone dressed). Prospects are in section 7. **Main pool:** the full seasons
2018-19 and 2021-22 to 2025-26. The two short seasons and a longer window (from 2011-12) are shown separately.
The by-season columns are the yardstick: how much the real mean and SD move from one season to the next.

In [11]:
rated = PG.real_ratings(E, range(2011, 2027))
r26 = rated[2026]
assert (r26["rating_war_60"] - E["before"]["rating_war_60"].reindex(r26.index)).abs().max() < 1e-12
real = PG.real_changes(E, rated, range(2011, 2026))
b = E["before"]
src = E["path"].set_index("playerId")["source"].reindex(b.index)
info = pd.DataFrame({"group": b["group"].to_numpy(), "age_group": PG.age_group(b["age"].to_numpy()).astype(str).to_numpy(),
                     "nhl": src.isin(PG.NHL_SOURCES).to_numpy(), "dressed": mt["dressed"].to_numpy(),
                     "minutes": mt["minutes"].to_numpy()})
sim_cols = np.flatnonzero(info["nhl"] & info["dressed"])
CH = many["changes"]
main = real[real["played"] & real["season"].isin(MAIN_SEASONS)].assign(age_group=lambda d: d["age_group"].astype(str))
SHOW = ["real_n", "sim_players", "real_mean", "sim_mean", "real_sd", "sim_sd", "real_pct_up3", "sim_pct_up3",
        "real_pct_down3", "sim_pct_down3", "real_mean_by_season", "real_sd_by_season"]
by_pos = PG.compare_table(main, CH[:, sim_cols], info.iloc[sim_cols], ["group"])
print("By position (whole-number OVR change; % = share of players):")
print(by_pos[["group"] + SHOW].round(2).to_string(index=False))

By position (whole-number OVR change; % = share of players):
group  real_n  sim_players  real_mean  sim_mean  real_sd  sim_sd  real_pct_up3  sim_pct_up3  real_pct_down3  sim_pct_down3 real_mean_by_season real_sd_by_season
    D    1433          185      -0.63     -0.08     2.85    2.75         12.63        15.09           23.59          17.28      -1.03 to -0.15      2.64 to 3.20
    F    2646          369      -0.59     -0.21     3.06    3.00         13.00        15.45           23.70          19.11      -1.56 to -0.09      2.74 to 3.19
    G     424           64      -0.01      0.01     2.49    2.71         14.15        17.56           14.15          16.45      -0.29 to +0.18      1.92 to 2.77


In [12]:
order = {g: i for i, g in enumerate(PG.AGE_GROUPS)}
by_age = PG.compare_table(main, CH[:, sim_cols], info.iloc[sim_cols], ["group", "age_group"])
by_age = by_age.sort_values(["group", "age_group"], key=lambda s: s.map(order) if s.name == "age_group" else s)
print("By position and age (age in season t):")
print(by_age[["group", "age_group"] + SHOW].round(2).to_string(index=False))
by_ageonly = PG.compare_table(main, CH[:, sim_cols], info.iloc[sim_cols], ["age_group"])
by_ageonly = by_ageonly.sort_values("age_group", key=lambda s: s.map(order))
print("\nBy age, all positions:")
print(by_ageonly[["age_group"] + SHOW].round(2).to_string(index=False))

By position and age (age in season t):
group age_group  real_n  sim_players  real_mean  sim_mean  real_sd  sim_sd  real_pct_up3  sim_pct_up3  real_pct_down3  sim_pct_down3 real_mean_by_season real_sd_by_season
    D  under 23     121           15      -0.92      0.06     2.65    2.79          7.44        18.05           24.79          15.21      -1.21 to -0.47      2.12 to 3.40
    D     23-26     469           46      -0.41      0.39     2.78    2.54         13.22        16.32           20.04          11.34      -1.16 to +0.26      2.47 to 3.10
    D     27-30     455           57      -0.67     -0.16     2.96    2.76         13.85        14.66           26.59          17.82      -1.14 to -0.11      2.39 to 3.69
    D       31+     388           67      -0.75     -0.36     2.83    2.83         12.11        13.96           23.97          21.36      -2.06 to -0.13      2.51 to 2.99
    F  under 23     294           46       0.19      0.59     2.56    2.93         17.69        24.33     

In [13]:
pl = real[real["played"]]
per = pl.groupby("season").apply(lambda d: PG.change_stats(d["change"]), include_groups=False)
per.index = [label(s) + (" (short)" if s in SHORT else "") for s in per.index]
print("Real, every season t (players who played):")
print(per.round(2).to_string())
longer = pl[~pl["season"].isin([2012] + SHORT)]
print("\nLonger window (2011-12 on, full seasons only) vs. the main pool vs. the simulation:")
rows = {"real, main pool": PG.change_stats(main["change"]), "real, 2011-12 on": PG.change_stats(longer["change"]),
        "real, short seasons": PG.change_stats(pl[pl["season"].isin(SHORT)]["change"]),
        "simulated": PG.change_stats(CH[:, sim_cols].ravel())}
print(pd.DataFrame(rows).T.round(2).to_string())

print("\nMinutes in season t (average per player):")
mins = pd.DataFrame({"real, main pool": main.groupby("group")["minutes"].mean(),
                     "simulated (dressed)": info.iloc[sim_cols].groupby("group")["minutes"].mean()})
mins["sim / real"] = mins["simulated (dressed)"] / mins["real, main pool"]
print(mins.round(2).to_string())

no_min_real = real[(real["minutes"] == 0) & real["season"].isin(MAIN_SEASONS)]
no_min_sim = np.flatnonzero(info["nhl"] & ~info["dressed"] & ~np.isnan(CH[0]))
print("\nNHL players with no minutes in season t (window rolls, regression, aging):")
print(pd.DataFrame({"real": PG.change_stats(no_min_real["change"]),
                    "simulated (not dressed)": PG.change_stats(CH[0, no_min_sim])}).T.round(2).to_string())

Real, every season t (players who played):
                     n  mean    sd  pct_up3  pct_down3
2011-12          697.0 -0.86  2.64     7.46      22.81
2012-13          636.0 -0.02  2.52    13.52      14.62
2013-14          694.0 -0.69  3.17    14.41      26.08
2014-15          696.0  0.01  2.91    17.10      16.95
2015-16          685.0 -0.61  2.48     8.32      21.46
2016-17          679.0 -0.75  2.69    10.60      24.01
2017-18          724.0 -0.76  2.90    11.05      23.34
2018-19          717.0 -0.41  2.85    14.09      21.48
2019-20 (short)  698.0  0.00  2.66    14.04      15.33
2020-21 (short)  698.0 -0.19  2.51    12.75      16.33
2021-22          760.0 -0.67  3.03    12.24      24.74
2022-23          763.0 -1.16  3.15    10.09      32.11
2023-24          765.0 -0.74  2.85    10.20      22.22
2024-25          750.0 -0.10  2.89    17.33      19.07
2025-26          748.0 -0.19  2.78    14.17      16.71

Longer window (2011-12 on, full seasons only) vs. the main pool vs. the simu

In [14]:
parts_real = PG.real_parts(E, rated, MAIN_SEASONS)
at_sim = many["maps"]["at"]["ovr_exact"].reindex(b.index)
nhl_d = info["nhl"].to_numpy() & info["dressed"].to_numpy()
sim_parts = pd.DataFrame({"group": b["group"].to_numpy()[nhl_d],
                          "aging_part": (at_sim - b["ovr_exact"]).to_numpy()[nhl_d],
                          "expected_exact_change": mt["expected_exact_change"].to_numpy()[nhl_d]})
sim_parts["season_part"] = sim_parts["expected_exact_change"] - sim_parts["aging_part"]
print("Where the average change comes from (exact OVR, players who played):")
print(pd.DataFrame({"real aging part": parts_real.groupby("group")["aging_part"].mean(),
                    "sim aging part": sim_parts.groupby("group")["aging_part"].mean(),
                    "real season part": parts_real.groupby("group")["season_part"].mean(),
                    "sim season part": sim_parts.groupby("group")["season_part"].mean()}).round(2).to_string())
gap = parts_real.groupby("group").apply(lambda d: np.average(d["season_vs_projection"], weights=d["minutes"]), include_groups=False)
print("\nReal season minus projection going into it (WAR/60, minutes-weighted), main pool:")
print(gap.map(lambda x: f"{x:+.4f}").to_string())
print("The simulated draws are centered on the projection, so their average is 0 by design.")

Where the average change comes from (exact OVR, players who played):
       real aging part  sim aging part  real season part  sim season part
group                                                                    
D                -0.08            0.02             -0.56            -0.12
F                -0.25           -0.07             -0.34            -0.16
G                -0.06           -0.13              0.06             0.08

Real season minus projection going into it (WAR/60, minutes-weighted), main pool:
group
D    -0.0055
F    -0.0024
G    +0.0007
The simulated draws are centered on the projection, so their average is 0 by design.


**Verdict: the spread matches real history; the average is about 0.4 OVR too high.**

- **Where it matches: the spread.**
  - **SD of the change:** forwards 3.00 simulated vs. 3.06 real, defensemen 2.75 vs. 2.85, goalies 2.71 vs. 2.49.
  - **All three are inside the real season-to-season range** (forwards 2.74 to 3.19, defensemen 2.64 to 3.20, goalies 1.92 to 2.77). All positions together: 2.90 simulated vs. 2.95 real (2.89 on the longer window from 2011-12).
  - **The age shape matches too:** young players go up, 31+ go down, in both.
  - **More minutes didn't widen it.** Simulated players play 20% to 36% more minutes than real ones (full health, 84 games), because the random draw was sized for a full season.
- **Where it doesn't: the average.**
  - **Overall:** real players who played moved -0.55 OVR a year on average, simulated ones -0.15. Forwards -0.59 vs. -0.21; defensemen -0.63 vs. -0.08; goalies about 0 in both.
  - **By age:** the simulation sits 0.35 to 0.57 higher in every age group.
  - **So:** +3 or more is a bit too common (15.6% vs. 13.0%) and -3 or worse too rare (18.3% vs. 22.8%).
  - **Size:** the simulated forward mean is inside the real season-to-season range (-1.56 to -0.09), but near its top; defensemen (-0.08) are just above it (-1.03 to -0.15).
- **The likely reason, from the split above:**
  - **Most of the gap is in the season part.** Real players who kept playing came in below their projection: forwards by 0.0024 WAR/60 a year, defensemen by 0.0055 (minutes-weighted). Real season parts are -0.34 OVR for forwards and -0.56 for defensemen, against -0.16 and -0.12 simulated.
  - **The simulation's draws are centered on each projection by design** (notebook 24). Its season part is slightly negative anyway (-0.12 to -0.16), because the OVR scale is flatter above 80 than below, so equal ups and downs in WAR/60 don't give equal OVR.
  - **A smaller piece is in the aging part** for forwards (-0.25 real vs. -0.07 simulated).
  - **Why might real seasons fall short?** Two candidates, not tested here:
    1. the projection runs a little high for returning players; notebook 18's training seasons showed the same sign;
    2. every real season, new players (rookies, call-ups) take a share of the league's WAR, and the simulation has no league-wide total.
- **Small groups to watch:** goalies 27-30 (simulated +0.73 vs. real -0.07, but only 21 simulated goalies) and defensemen under 23 (+0.06 vs. -0.92, 15 simulated players).
- **NHL players with no minutes:** -0.31 simulated (SD 2.02) vs. -0.55 real (SD 2.29). Same direction, a little milder.
- **Nothing was tuned.** If you want the averages to match, the smallest fix would be a league-wide pull on the season part of about -0.4 OVR (about -0.002 WAR/60 for forwards, -0.005 for defensemen). I haven't applied it: it's your call, and it would change what "centered on the projection" means for the simulator.

## 7. Prospects: what a real year does to a prospect (and the rule adopted)

For a prospect who isn't dressed, the engine itself has nothing new: same pre-NHL stats, one year older. Because the
prior has a negative age term, forwards drift down a little. In real life he would also play a new AHL, junior or
European season. So here is what **real** prospects did in a year, and the rule built from it.

**How:** for each season t, notebook 19's prospect list going into t (drafted, not on the NHL path, played somewhere
in the season before, 24 or younger), kept only if he played **no NHL minutes in t**. Each is rated going into t, then
going into t+1 two ways:

- **real:** adding his real season-t pre-NHL season (NHLe or save%) from the profiles;
- **engine:** with no new season, exactly what the progression engine does.

**Coverage:** the draft lists on file start in 2019, so in earlier seasons the older prospects (drafted before 2019)
are missing. Every 2019-on pick has a profile, whatever happened to him, so there's no "only the ones who made it"
bias within those classes. Seasons 2024-25 and 2025-26 cover every age group: those two are the main pool.

In [15]:
pros = PG.real_prospect_changes(E, range(2020, 2026))
cov = pros.groupby("season").agg(prospects=("playerId", "size"), oldest_draft=("draft_year", "min"),
                                 age_24_plus=("age", lambda a: int((a >= 24).sum())))
cov.index = [label(s) for s in cov.index]
print("Prospects with no NHL minutes in t, by season t:")
print(cov.to_string())
PMAIN = [2024, 2025]
pm = pros[pros["season"].isin(PMAIN)]
def pstats(d):
    return pd.Series({"n": len(d), "real_mean": d["change_real"].mean(), "real_sd": d["change_real"].std(ddof=0),
                      "real_pct_up3": (d["change_real"] >= 3).mean() * 100, "real_pct_down3": (d["change_real"] <= -3).mean() * 100,
                      "engine_mean": d["change_engine"].mean(), "engine_sd": d["change_engine"].std(ddof=0),
                      "engine_exact_mean": d["exact_engine"].mean()})
print("\nMain pool (2024-25 and 2025-26), by position and age:")
print(pm.groupby(["group", "age_group"], observed=True).apply(pstats, include_groups=False).round(2).to_string())
print("\nAll seasons 2020-21 on, by position:")
print(pros.groupby("group").apply(pstats, include_groups=False).round(2).to_string())

3 pick(s) without a playerId dropped
Prospects with no NHL minutes in t, by season t:
         prospects  oldest_draft  age_24_plus
2020-21        406          2019            0
2021-22        537          2019            0
2022-23        732          2019            5
2023-24        864          2019           10
2024-25        998          2019           60
2025-26       1051          2019          110

Main pool (2024-25 and 2025-26), by position and age:
                     n  real_mean  real_sd  real_pct_up3  real_pct_down3  engine_mean  engine_sd  engine_exact_mean
group age_group                                                                                                    
D     18-19      208.0       0.96     1.43         11.54            0.00        -0.02       0.14              -0.04
      20-21      236.0      -0.09     1.59          5.51            7.63        -0.02       0.14              -0.04
      22-23      178.0      -0.40     1.35          1.12            5.62 

**The rule adopted (Ryan's decision):** a prospect with no NHL minutes gets his OVR before + the real average exact
change of a prospect year for his age group and position, from the main pool above (`E["expected_dev"]`, built in
`load_engine`). His rating WAR/60 follows the new OVR through the locked scale (forwards and goalies: his projection;
defensemen: his ranking prior, while his projection stays the position average, as today). The engine's own number
stays in `engine_ovr_after`.

In [16]:
assert E["expected_dev"].round(12).equals(PG.expected_development(pm).round(12))
print("Expected development (exact OVR), built in the engine from 2024-25 and 2025-26:")
print(E["expected_dev"].round(2).to_string())
o = prog[prog["flag"].str.startswith("prospect, no NHL minutes")]
key = list(zip(o["group"], PG.prospect_age_group(o["age"].to_numpy()).astype(str)))
want = np.array([E["expected_dev"]["expected_change"].get(k, np.nan) for k in key])
gap = np.nanmax(np.abs(o["exact_change"].to_numpy() - want))
print(f"\nCheck: each developed prospect's exact change = his expected development (largest gap {gap:.1e})")
assert gap < 1e-9
print(f"\nIn the opening-night story, {len(o):,} prospects have no NHL minutes. Average whole-number change:")
print(pd.DataFrame({"engine (no new season)": o.groupby("group")["engine_change"].mean(),
                    "adopted (expected development)": o.groupby("group")["change"].mean(),
                    "OVRs that differ": o.assign(d=o["change"] != o["engine_change"]).groupby("group")["d"].sum()}).round(2).to_string())
print(f"\n{TEAM}'s prospects with no NHL minutes:")
print(o[o["team"] == TEAM][["name", "position", "age", "ovr_before", "engine_ovr_after", "ovr_after", "change"]].to_string(index=False))

Expected development (exact OVR), built in the engine from 2024-25 and 2025-26:
                 expected_change  prospect_seasons
group age_group                                   
D     18-19                 0.92               208
      20-21                -0.11               236
      22-23                -0.41               178
      24+                  -0.14                47
F     18-19                 0.98               338
      20-21                -0.29               385
      22-23                -1.14               322
      24+                  -0.47               108
G     18-19                -0.08                65
      20-21                -0.19                79
      22-23                -0.01                68
      24+                  -0.34                15

Check: each developed prospect's exact change = his expected development (largest gap 7.1e-15)

In the opening-night story, 1,153 prospects have no NHL minutes. Average whole-number change:
       engine (

**What real prospect years look like, next to the engine:**

- **Real prospects move, in both directions.**
  - **Forwards:** 18-19 year-olds gain +0.96 OVR on average (29% gain 3 or more); 20-21 lose -0.28; 22-23 lose -1.16 (26% drop 3 or more). The spread is wide (SD about 2.4 to 3.1).
  - **Defensemen** move less (SD about 1.4 to 1.6): +0.96 at 18-19, about 0 to -0.4 after.
  - **Goalie prospects** barely move (SD about 1.2), close to 0 on average.
  - **Why the late-age drop:** a prospect still in junior or the AHL at 22-23 is, on average, one who isn't progressing.
- **The engine's "no new season" number is too flat.**
  - **Forwards:** -0.55 to -0.62 at every age, the prior's age term only.
  - **Defensemen and goalies:** about 0 (their priors have almost no age term), except goalies 24+ (-0.67).
  - **So it's wrong both ways:** 1.5 OVR too low for 18-19 forwards and 0.6 too high for 22-23. It has almost no spread (SD 0.5), against a real SD near 3.
- **The rule adopted (Ryan's decision):** add the real average change for his age group and position, flagged "expected development, not simulated".
  - **The average:** in this story it moves the 1,153 non-playing prospects from -0.57 to -0.17 (forwards), -0.05 to +0.08 (defensemen) and -0.12 to -0.11 (goalies). 490 of their OVRs differ from the engine's number, by a point or two. Each one's exact change equals his expected development (checked).
  - **What it fixes:** the average by age (young forwards now rise, older ones fall, as in real life).
  - **What it doesn't:** the spread. Every 19-year-old forward gets the same +1, while real ones spread widely (SD about 2.9). A fuller version would draw each prospect's development from the real spread for his age and position, the way NHL players get a random season. That's a bigger step for a later notebook.

## 8. Speed and git

The engine is built once per session; after that, one story's progression and the 10,000-season version are fast.
Nothing is written. A reserved path for a future saved progression table is checked against .gitignore.

In [17]:
print("Speed (seconds):")
for k, v in SPEED.items():
    print(f"  {k}: {v:.1f}")

def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd="..").stdout.rstrip()

print("\n" + git("check-ignore", "-v", "data/progression_2026-27.csv"))
tracked = git("ls-files", "data").split("\n")
print("Tracked files under data/:", tracked)
assert tracked == ["data/README.md"]
print("\ngit status --porcelain:")
print(git("status", "--porcelain"))

Speed (seconds):
  build the engine (once): 18.8
  one story (notebook 30): 0.6
  one story's progression: 1.9
  10,000 seasons' progression (incl. the 10,000-season run): 12.3

.gitignore:2:data/*.csv	data/progression_2026-27.csv
Tracked files under data/: ['data/README.md']

git status --porcelain:
 M CLAUDE.md
 M src/development.py
 M src/franchise.py
?? notebooks/31_ovr_progression.ipynb
?? src/progression.py


## Takeaways

- **The engine reproduces today's ratings exactly.** Run on the real history that made them, `project_season` gives all 2,503 players their 2026-27 projection and OVR: NHL skaters and goalies, skater and goalie prospects, and new players. The largest gap is 1e-16, and no OVR differs.
- **One story, one new rating per player.**
  - **In this story:** from opening night (seed 0), 640 players played NHL minutes. Their changes average about 0 (forwards -0.05, defensemen +0.40, goalies -0.38) with an SD of about 3 OVR (2.4 for goalies), from -11 to +12.
  - **The parts add up exactly:** aging + season = the exact change.
  - **The straight-line shortcut matches** the direct re-projection to 2e-16.
  - **Zero change holds:** 306 players lose no season from their window, and they change by exactly 0.
- **The big movers make hockey sense.**
  - **Players with little NHL history** move most, because their rating rested on a prior (Connors -11, Stromgren -9, Brisebois +10).
  - **Veterans fall** with age and a good 2023-24 leaving the window (Marchessault -10, Martinook -8).
  - **Breakouts and busts** account for the rest (Dach +8, Ferraro -7). Stars barely move.
- **Two pieces hide inside the "aging part":**
  - **the window roll:** 882 players lose a 2023-24 season, worth +0.4 OVR on average for skaters, SD about 2;
  - **a path switch for prospect defensemen who dress:** they were rated on the ranking prior but played at the position average, so the 7 who dress jump +7.6 on average, +4.1 of it in the aging part.
- **10,000 seasons:**
  - **Speed:** 10.4 s in all, including the 10,000-season run. One story's progression takes 1.7 s; the engine builds once in about 18 s, including measuring prospect development (much longer the first time after a restart, while the files are read from disk).
  - **Detroit from opening night:** Wallinder (+3.7, 62% chance of +3) and Anton Johansson (+3.6) are the likeliest risers; Rasmussen (-2.5, 48% chance of -3 or worse) and Larkin (-1.6) the likeliest fallers.
  - **Detroit from today with the update:** Wallinder +5.5 (78%) and Rasmussen -4.1 (69%): the update has already seen their real starts.
- **Calibration against real history:**
  - **The spread matches:** SD 2.90 simulated vs. 2.95 real, inside the real season-to-season range for every position.
  - **The average is about 0.4 OVR too high:** -0.15 vs. -0.55 a year. So +3 or more is a bit too common (15.6% vs. 13.0%) and -3 or worse too rare (18.3% vs. 22.8%).
  - **The gap sits mostly in the season part:** real returning players came in 0.0024 (forwards) and 0.0055 (defensemen) WAR/60 below their projections, while the simulated draws are centered by design.
  - **Not tuned.** The smallest fix (a league-wide pull of about -0.4 OVR on the season part) is described for you to decide.
- **Prospects (rule adopted):**
  - **Real development** depends strongly on age: forwards +0.96 at 18-19, -1.16 at 22-23, with an SD near 3.
  - **The engine's own "no new season" number missed this:** a flat -0.57 for forwards, about 0 for others, almost no spread. It stays in `engine_ovr_after`.
  - **Adopted:** a prospect with no NHL minutes gets the real average change for his age group and position (2024-25 and 2025-26), flagged "expected development, not simulated". It fixes the average, not the spread.
- **For in-season stories,** the drawn season WAR/60 covers the whole season, real games included. The update's draw has already seen the real goals and assists, and there's no real WAR for a partial season. 222 players with real games who aren't in a lineup now are flagged; their real games aren't counted.
- **Files:**
  - **New:** `src/progression.py`.
  - **Add-only changes:** `development.nb18_skater_model` and `project_skater_prospects(season=)` in development.py; `franchise.play(keep_z=)`. Defaults are unchanged, and today's ratings reproduce exactly with them.
  - **Nothing is written.** The reserved path data/progression_2026-27.csv is ignored by git, and only data/README.md is tracked under data/.